# Intermediate 11 — Hypothesis Testing: Neyman-Pearson and Likelihood Ratio Tests

Practice notebook for the **"Hypothesis Testing: Neyman-Pearson and Likelihood Ratio Tests"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


## Part 1 — Neyman-Pearson lemma: most powerful test (simple vs. simple)

The PDF states the **Neyman-Pearson lemma**: to test

$$
H_0:\theta=\theta_0 \quad \text{vs.}\quad H_1:\theta=\theta_1,
$$

the **most powerful** test at level $\alpha$ rejects $H_0$ for **small values** of the
likelihood ratio

$$
\Lambda(x)=\frac{f(x;\theta_0)}{f(x;\theta_1)}.
$$

Equivalently one works with $-2\log\Lambda$, rejecting for large values.

We instantiate the lemma on $n$ i.i.d. observations with

$$
H_0: X_i\sim\mathcal N(0,1) \quad \text{vs.}\quad H_1: X_i\sim\mathcal N(1,1).
$$

With $S=\sum_i X_i$ the likelihood ratio factorizes and the rejection region is
$S>c$. Under $H_0$, $S\sim\mathcal N(0,n)$; under $H_1$, $S\sim\mathcal N(n,n)$. The
level-$\alpha$ threshold is

$$
c = \sqrt{n}\,z_{1-\alpha}, \qquad z_{1-\alpha}=\Phi^{-1}(1-\alpha).
$$

We build the test, then verify by simulation that it (a) attains the nominal size and
(b) is **strictly more powerful** than any other level-$\alpha$ test — here a
deliberately two-sided test $|S|>c'$ calibrated to the same size.


In [ ]:
# Neyman-Pearson most-powerful test for H0: N(0,1) vs H1: N(1,1), n observations
n = 10
alpha = 0.05
z = stats.norm.ppf(1 - alpha)          # z_{1-alpha}
c_np = np.sqrt(n) * z                 # NP threshold on S = sum(X_i)

# Theoretical size and power
size_th = 1 - stats.norm.cdf(c_np, loc=0, scale=np.sqrt(n))     # P_0(S > c)
power_th = 1 - stats.norm.cdf(c_np, loc=n, scale=np.sqrt(n))    # P_1(S > c)
print(f"NP threshold c_np        = {c_np:.4f}")
print(f"Theory size  P_0(reject)  = {size_th:.4f}  (target alpha = {alpha})")
print(f"Theory power P_1(reject)  = {power_th:.4f}")

# --- Simulation: size under H0, power under H1 -----------------------------
rng = np.random.default_rng(11)
Nsim = 200_000
S0 = rng.normal(0, np.sqrt(n), size=Nsim)   # S under H0
S1 = rng.normal(n, np.sqrt(n), size=Nsim)   # S under H1

size_sim = np.mean(S0 > c_np)
power_sim = np.mean(S1 > c_np)
print()
print(f"Sim size   = {size_sim:.4f}")
print(f"Sim power  = {power_sim:.4f}")

# --- A different level-alpha test: two-sided |S| > c' ---------------------
# Under H0, S~N(0,n); choose c' s.t. P_0(|S|>c') = alpha  =>  c' = sqrt(n)*z_{1-alpha/2}
c_two = np.sqrt(n) * stats.norm.ppf(1 - alpha / 2)
size_two = np.mean(np.abs(S0) > c_two)
power_two = np.mean(np.abs(S1) > c_two)
print()
print(f"Two-sided threshold c'    = {c_two:.4f}")
print(f"Two-sided sim size       = {size_two:.4f}  (target {alpha})")
print(f"Two-sided sim power      = {power_two:.4f}")
print(f"NP power advantage       = {power_sim - power_two:+.4f}")


## Part 2 — Visualizing the likelihood ratio and the ROC

For the same simple-vs-simple problem, the likelihood ratio for a single observation
$x$ is

$$
\Lambda(x)=\frac{f_0(x)}{f_1(x)}=\exp\!\left(-x+\tfrac12\right),
$$

so $\Lambda$ is **monotone decreasing** in $x$ — rejecting for small $\Lambda$ is the same
as rejecting for large $x$. The **trade-off** between size $P_0(\text{reject})$ and power
$P_1(\text{reject})$ as we sweep the threshold traces the test's **ROC curve**. The
Neyman-Pearson test is, by construction, the point on this curve with the largest power
at the chosen size $\alpha$.

**Your turn:** change $\mu_1$ below from $1$ to $0.3$ (a weaker signal). How does the
ROC curve collapse toward the diagonal?


In [ ]:
mu0, mu1 = 0.0, 1.0
sigma = 1.0
x = np.linspace(-3, 5, 600)
f0 = stats.norm.pdf(x, mu0, sigma)
f1 = stats.norm.pdf(x, mu1, sigma)
Lambda = f0 / f1

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: densities and the LR
axL.plot(x, f0, label=r"$f_0$  $\mathcal N(0,1)$")
axL.plot(x, f1, label=r"$f_1$  $\mathcal N(1,1)$")
axL.axvline(stats.norm.ppf(0.95), color="r", linestyle="--",
            label=r"$c=z_{0.95}\approx$"+f"{stats.norm.ppf(0.95):.3f}")
axL.set_title("Densities and NP threshold")
axL.legend(); axL.set_xlabel("$x$")

# Right: ROC by sweeping threshold c on x
thr = np.linspace(-4, 5, 400)
size = 1 - stats.norm.cdf(thr, mu0, sigma)
power = 1 - stats.norm.cdf(thr, mu1, sigma)
axR.plot(size, power, label="NP test ROC")
axR.plot([0, 1], [0, 1], "k--", alpha=0.5, label="random classifier")
# Mark the alpha=0.05 operating point
c05 = stats.norm.ppf(0.95)
axR.plot(1-stats.norm.cdf(c05, mu0, sigma), 1-stats.norm.cdf(c05, mu1, sigma),
         "ro", label=r"$\alpha=0.05$")
axR.set_xlabel(r"size  $P_0(\mathrm{reject})$")
axR.set_ylabel(r"power $P_1(\mathrm{reject})$")
axR.set_title("ROC of the NP test")
axR.legend()
plt.tight_layout(); plt.show()

print(f"At alpha=0.05: size={1-stats.norm.cdf(c05,mu0,sigma):.4f}, "
      f"power={1-stats.norm.cdf(c05,mu1,sigma):.4f}")


## Part 3 — Likelihood ratio test for a composite hypothesis (Wilks' theorem)

For a **composite** hypothesis $H_0:\theta\in\Theta_0$ vs. $H_1:\theta\in\Theta$,
the PDF defines

$$
\Lambda(x)=\frac{\sup_{\theta\in\Theta_0} L(\theta)}{\sup_{\theta\in\Theta} L(\theta)},
$$

rejecting $H_0$ for **small** $\Lambda$. Under regularity, **Wilks' theorem** gives

$$
-2\log\Lambda(X)\;\overset{\text{approx}}{\sim}\;\chi^2_k,
$$

where $k=\dim(\Theta)-\dim(\Theta_0)$ is the drop in free parameters.

We test $H_0:\mu=0$ vs. $H_1:\mu\ne 0$ with **known** $\sigma^2=1$, $n$ i.i.d.
observations. The unrestricted MLE is $\hat\mu=\bar X$; under $H_0$ the supremum is at
$\mu=0$. A short calculation gives

$$
-2\log\Lambda = n\,\bar X^{2} \;\overset{H_0}{\sim}\; \chi^2_1.
$$

We verify this distributional result by Monte Carlo: draw many datasets under $H_0$,
compute $-2\log\Lambda$ for each, and compare the histogram to $\chi^2_1$.


In [ ]:
n = 30
sigma2 = 1.0
rng = np.random.default_rng(123)
Nsim = 50_000

# Draw Nsim datasets of size n under H0: N(0,1)
X = rng.normal(0, np.sqrt(sigma2), size=(Nsim, n))
xbar = X.mean(axis=1)

# -2 log Lambda = n * xbar^2 (sigma2 known = 1)
neg2logL = n * xbar ** 2

# Compare empirical quantiles to chi2_1 quantiles (QQ plot + moments)
q_the = stats.chi2.ppf(np.linspace(0.001, 0.999, 200), df=1)
q_emp = np.quantile(neg2logL, np.linspace(0.001, 0.999, 200))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].hist(neg2logL, bins=80, density=True, alpha=0.6, label=r"sim $-2\log\Lambda$")
xx = np.linspace(0, neg2logL.max(), 400)
ax[0].plot(xx, stats.chi2.pdf(xx, df=1), "r-", lw=2, label=r"$\chi^2_1$")
ax[0].set_title(r"$-2\log\Lambda$ under $H_0$ vs $\chi^2_1$")
ax[0].legend()

ax[1].plot(q_the, q_emp, ".", label="sim vs theory")
ax[1].plot(q_the, q_the, "k--", alpha=0.5)
ax[1].set_xlabel(r"$\chi^2_1$ quantiles")
ax[1].set_ylabel("simulated quantiles")
ax[1].set_title("QQ plot"); ax[1].legend()
plt.tight_layout(); plt.show()

# Nominal size check: reject H0 if -2logL > chi2_1 critical at alpha
alpha = 0.05
crit = stats.chi2.ppf(1 - alpha, df=1)
sim_size = np.mean(neg2logL > crit)
print(f"chi2_1 95%-critical = {crit:.4f}")
print(f"sim size  P_0(reject) = {sim_size:.4f}  (target {alpha})")
print(f"mean(-2logLambda) = {neg2logL.mean():.4f}  vs chi2_1 mean = 1")


## Part 4 — The t-test as a likelihood ratio test (unknown variance)

The PDF's example: with $X_i\stackrel{iid}{\sim}\mathcal N(\mu,\sigma^2)$ and
**both** $\mu,\sigma^2$ unknown, test $H_0:\mu=\mu_0$ vs. $H_1:\mu\ne\mu_0$.

The unrestricted MLEs are $(\hat\mu,\hat\sigma^2)=(\bar X, \tfrac1n\sum(X_i-\bar X)^2)$.
Under $H_0$ we fix $\mu=\mu_0$ and maximize over $\sigma^2$ only. Working out
$-2\log\Lambda$ gives a function of the t-statistic

$$
T = \frac{\bar X - \mu_0}{S/\sqrt{n}}, \qquad
-2\log\Lambda = n\,\log\!\left(1 + \frac{T^{2}}{n-1}\right).
$$

For large $n$, $\log(1+T^2/(n-1))\approx T^2/n$, so $-2\log\Lambda\to T^2$. Under $H_0$,
$T\sim t_{n-1}$ so $T^2\sim F_{1,n-1}$, and as $n\to\infty$ this tends to $\chi^2_1$ —
consistent with Wilks' $k=1$ (one parameter, $\mu$, restricted).

We simulate under $H_0$ and show both: (a) $-2\log\Lambda$ matches $\chi^2_1$ for moderate
$n$, and (b) the exact $T^2$ matches $F_{1,n-1}$.


In [ ]:
n = 40
mu0 = 0.0
rng = np.random.default_rng(2024)
Nsim = 50_000

X = rng.normal(mu0, 1.0, size=(Nsim, n))
xbar = X.mean(axis=1)
S2 = X.var(axis=1, ddof=1)            # sample variance (unbiased)
T = (xbar - mu0) / np.sqrt(S2 / n)    # t-statistic
neg2logL = n * np.log(1 + T ** 2 / (n - 1))

alpha = 0.05
crit_chi2 = stats.chi2.ppf(1 - alpha, df=1)
crit_F = stats.f.ppf(1 - alpha, dfn=1, dfd=n - 1)   # exact T^2 critical
crit_t = stats.t.ppf(1 - alpha / 2, df=n - 1)       # |T| critical

size_wilks = np.mean(neg2logL > crit_chi2)
size_F = np.mean(T ** 2 > crit_F)
size_t = np.mean(np.abs(T) > crit_t)

print(f"Wilks  (-2logLambda > chi2_1 crit)   size = {size_wilks:.4f}  (target {alpha})")
print(f"Exact  (T^2 > F_{{1,{n-1}}} crit)      size = {size_F:.4f}")
print(f"Exact  (|T| > t_{{{n-1}}} crit)        size = {size_t:.4f}")
print()
print(f"mean(-2logLambda) = {neg2logL.mean():.4f}  (chi2_1 mean = 1)")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].hist(neg2logL, bins=80, density=True, alpha=0.6, label=r"$-2\log\Lambda$")
xx = np.linspace(0, 10, 400)
ax[0].plot(xx, stats.chi2.pdf(xx, df=1), "r-", lw=2, label=r"$\chi^2_1$ (Wilks)")
ax[0].set_title(f"n={n}: Wilks already close to chi2_1")
ax[0].legend()

ax[1].hist(T ** 2, bins=80, density=True, alpha=0.6, label=r"$T^2$")
ax[1].plot(xx, stats.f.pdf(xx, dfn=1, dfd=n - 1), "g-", lw=2,
           label=rf"$F_{{1,{n-1}}}$ (exact)")
ax[1].set_xlim(0, 10)
ax[1].set_title("Exact finite-n distribution of T^2")
ax[1].legend()
plt.tight_layout(); plt.show()


## Part 5 — Power of the LRT as $\mu$ moves away from $\mu_0$

Wilks' theorem describes the **null** distribution of $-2\log\Lambda$. Under a fixed
alternative $\mu\ne\mu_0$, $-2\log\Lambda$ is **non-central** $\chi^2$ (a
$\chi^2_1(\lambda)$ with non-centrality $\lambda=n(\mu-\mu_0)^2/\sigma^2$ for the
known-variance case). We sweep $\mu$ and plot the empirical rejection rate of the
$\alpha=0.05$ LRT — the **power curve** — and overlay the non-central $\chi^2$ theory.

**Your turn:** bump $n$ from 30 to 100. Does the power curve get steeper around
$\mu_0=0$? Why?


In [ ]:
n = 30
sigma2 = 1.0
alpha = 0.05
crit = stats.chi2.ppf(1 - alpha, df=1)
rng = np.random.default_rng(7)
Nsim = 20_000

mu_grid = np.linspace(-1.0, 1.0, 21)
power_sim = []
power_th = []
for mu in mu_grid:
    X = rng.normal(mu, np.sqrt(sigma2), size=(Nsim, n))
    xb = X.mean(axis=1)
    stat = n * xb ** 2                      # -2logLambda (sigma2 known)
    power_sim.append(np.mean(stat > crit))
    lam_ncp = n * (mu - 0.0) ** 2 / sigma2  # non-centrality
    power_th.append(1 - stats.ncx2.cdf(crit, df=1, nc=lam_ncp))

plt.plot(mu_grid, power_sim, "o-", label="simulated power")
plt.plot(mu_grid, power_th, "k--", label=r"non-central $\chi^2_1$ theory")
plt.axvline(0, color="grey", linestyle=":", alpha=0.6)
plt.axhline(alpha, color="r", linestyle=":", alpha=0.6, label=f"alpha={alpha}")
plt.xlabel(r"$\mu$")
plt.ylabel("power")
plt.title(rf"LRT power curve for $H_0:\mu=0$ (n={n}, known $\sigma^2$)")
plt.legend(); plt.tight_layout(); plt.show()


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. For a single observation $X$ testing $H_0:X\sim\mathcal N(0,1)$ vs. $H_1:X\sim\mathcal N(\mu_1,1)$ with $\mu_1>0$, derive the NP rejection region at level $\alpha$ and give a closed-form expression for the power as a function of $\mu_1$ and $\alpha$.

2. Let $X_1,\dots,X_n\stackrel{iid}{\sim}\mathcal N(\mu,1)$ with $\mu$ unknown but $\sigma^2=1$ known. Write down the likelihood ratio statistic $\Lambda$ for $H_0:\mu=0$ vs. $H_1:\mu\ne 0$ and show $-2\log\Lambda=n\bar X^2\sim\chi^2_1$ under $H_0$.

3. For the unknown-variance test of $H_0:\mu=\mu_0$ in $\mathcal N(\mu,\sigma^2)$, show that $-2\log\Lambda=n\log\!\bigl(1+\frac{T^2}{n-1}\bigr)$ where $T=\frac{\bar X-\mu_0}{S/\sqrt{n}}$, and argue that it converges to $T^2$ and hence to $\chi^2_1$ for large $n$.

4. In Part 1 the two-sided test $|S|>c'$ was calibrated to the same $\alpha=0.05$ as the NP test but had lower power. Explain in one sentence why this does **not** contradict the Neyman-Pearson lemma.

5. State Wilks' theorem precisely (parameters, dimensions, conditions) and identify $k$ for the test $H_0:\mu=\mu_0,\,\sigma^2\text{ free}$ vs. $H_1:(\mu,\sigma^2)\text{ free}$ in the normal model.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** $\Lambda(x)=f_0(x)/f_1(x)=\exp\!\bigl(-(x^2-(x-\mu_1)^2)/2\bigr)=\exp(-\mu_1 x+\mu_1^2/2)$, monotone decreasing in $x$ (since $\mu_1>0$). Reject for $x>c$ with $c=z_{1-\alpha}$ so $P_0(\text{reject})=\alpha$. Power $=P_1(X>c)=1-\Phi(c-\mu_1)=1-\Phi(z_{1-\alpha}-\mu_1)$.

**2.** $L(\mu)\propto\exp\!\bigl(-\tfrac12\sum(X_i-\mu)^2\bigr)$. Unrestricted MLE $\hat\mu=\bar X$; restricted supremum at $\mu=0$. So $\Lambda=L(0)/L(\bar X)=\exp\!\bigl(-\tfrac12\bigl[\sum X_i^2 - \sum(X_i-\bar X)^2\bigr]\bigr)=\exp\!\bigl(-\tfrac{n}{2}\bar X^2\bigr)$, giving $-2\log\Lambda=n\bar X^2$. Under $H_0$, $\bar X\sim\mathcal N(0,1/n)$ so $n\bar X^2\sim\chi^2_1$.

**3.** With $\hat\sigma^2_0=\tfrac1n\sum(X_i-\mu_0)^2=\tfrac1n\bigl[\sum(X_i-\bar X)^2+n(\bar X-\mu_0)^2\bigr]=S^2\tfrac{n-1}{n}+(\bar X-\mu_0)^2$ and $\hat\sigma^2=\tfrac{n-1}{n}S^2$, the ratio $\hat\sigma^2_0/\hat\sigma^2=1+\frac{(\bar X-\mu_0)^2}{S^2(n-1)/n}=1+\frac{T^2}{n-1}$. Hence $-2\log\Lambda=n\log(1+T^2/(n-1))$. For large $n$, $\log(1+u)\approx u$ with $u=T^2/(n-1)\to 0$ in probability, so $-2\log\Lambda\to T^2$; and $T\to\mathcal N(0,1)$ so $T^2\to\chi^2_1$.

**4.** The Neyman-Pearson lemma guarantees **most power among all level-$\alpha$ tests** for a simple-vs-simple problem; the two-sided test is a different test that also has size $\alpha$ but is **not** the NP-likelihood-ratio test for the simple alternative $\mu=1$, so it can (and here does) have lower power at that specific alternative.

**5.** Wilks: under $H_0$ and regularity, $-2\log\Lambda\overset{d}{\to}\chi^2_k$ with $k=\dim\Theta-\dim\Theta_0$. Here $\dim\Theta=2$ (parameters $\mu,\sigma^2$), $\dim\Theta_0=1$ (only $\sigma^2$ free, $\mu$ fixed), so $k=1$.

</details>
